In [11]:
import numpy as np

**Module** is an abstract class which defines fundamental methods necessary for a training a neural network. You do not need to change anything here, just read the comments.

In [12]:
class Module(object):
    """
    Basically, you can think of a module as of a something (black box)
    which can process `input` data and produce `ouput` data.
    This is like applying a function which is called `forward`:

        output = module.forward(input)

    The module should be able to perform a backward pass: to differentiate the `forward` function.
    More, it should be able to differentiate it if is a part of chain (chain rule).
    The latter implies there is a gradient from previous step of a chain rule.

        gradInput = module.backward(input, gradOutput)
    """
    def __init__ (self):
        self.output = None
        self.gradInput = None
        self.training = True

    def forward(self, input):
        """
        Takes an input object, and computes the corresponding output of the module.
        """
        return self.updateOutput(input)

    def backward(self,input, gradOutput):
        """
        Performs a backpropagation step through the module, with respect to the given input.

        This includes
         - computing a gradient w.r.t. `input` (is needed for further backprop),
         - computing a gradient w.r.t. parameters (to update parameters while optimizing).
        """
        self.updateGradInput(input, gradOutput)
        self.accGradParameters(input, gradOutput)
        return self.gradInput


    def updateOutput(self, input):
        """
        Computes the output using the current parameter set of the class and input.
        This function returns the result which is stored in the `output` field.

        Make sure to both store the data in `output` field and return it.
        """

        # The easiest case:

        # self.output = input
        # return self.output

        pass

    def updateGradInput(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own input.
        This is returned in `gradInput`. Also, the `gradInput` state variable is updated accordingly.

        The shape of `gradInput` is always the same as the shape of `input`.

        Make sure to both store the gradients in `gradInput` field and return it.
        """

        # The easiest case:

        # self.gradInput = gradOutput
        # return self.gradInput

        pass

    def accGradParameters(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own parameters.
        No need to override if module has no parameters (e.g. ReLU).
        """
        pass

    def zeroGradParameters(self):
        """
        Zeroes `gradParams` variable if the module has params.
        """
        pass

    def getParameters(self):
        """
        Returns a list with its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def getGradParameters(self):
        """
        Returns a list with gradients with respect to its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def train(self):
        """
        Sets training mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = True

    def evaluate(self):
        """
        Sets evaluation mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = False

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Module"

# Sequential container

**Define** a forward and backward pass procedures.

In [13]:
class Sequential(Module):
    """
         This class implements a container, which processes `input` data sequentially.

         `input` is processed by each module (layer) in self.modules consecutively.
         The resulting array is called `output`.
    """

    def __init__ (self):
        super(Sequential, self).__init__()
        self.modules = []

    def add(self, module):
        """
        Adds a module to the container.
        """
        self.modules.append(module)

    def updateOutput(self, input):
        """
        Basic workflow of FORWARD PASS:

            y_0    = module[0].forward(input)
            y_1    = module[1].forward(y_0)
            ...
            output = module[n-1].forward(y_{n-2})


        Just write a little loop.
        """

        # Your code goes here. ################################################
        current_output = input

        for module in self.modules:
            current_output = module.forward(current_output)

        self.output = current_output
        return self.output

    def backward(self, input, gradOutput):
        """
        Workflow of BACKWARD PASS:

            g_{n-1} = module[n-1].backward(y_{n-2}, gradOutput)
            g_{n-2} = module[n-2].backward(y_{n-3}, g_{n-1})
            ...
            g_1 = module[1].backward(y_0, g_2)
            gradInput = module[0].backward(input, g_1)


        !!!

        To ech module you need to provide the input, module saw while forward pass,
        it is used while computing gradients.
        Make sure that the input for `i-th` layer the output of `module[i]` (just the same input as in forward pass)
        and NOT `input` to this Sequential module.

        !!!

        """
        # Your code goes here. ################################################
        current_grad = gradOutput

        for i in range(len(self.modules) - 1, -1, -1):
            if i == 0:
                module_input = input
            else:
                module_input = self.modules[i - 1].output

            current_grad = self.modules[i].backward(module_input, current_grad)

        self.gradInput = current_grad
        return self.gradInput


    def zeroGradParameters(self):
        for module in self.modules:
            module.zeroGradParameters()

    def getParameters(self):
        """
        Should gather all parameters in a list.
        """
        return [x.getParameters() for x in self.modules]

    def getGradParameters(self):
        """
        Should gather all gradients w.r.t parameters in a list.
        """
        return [x.getGradParameters() for x in self.modules]

    def __repr__(self):
        string = "".join([str(x) + '\n' for x in self.modules])
        return string

    def __getitem__(self,x):
        return self.modules.__getitem__(x)

    def train(self):
        """
        Propagates training parameter through all modules
        """
        self.training = True
        for module in self.modules:
            module.train()

    def evaluate(self):
        """
        Propagates training parameter through all modules
        """
        self.training = False
        for module in self.modules:
            module.evaluate()

# Layers

## 1 (0.2). Linear transform layer
Also known as dense layer, fully-connected layer, FC-layer, InnerProductLayer (in caffe), affine transform
- input:   **`batch_size x n_feats1`**
- output: **`batch_size x n_feats2`**

In [14]:
class Linear(Module):
    """
    A module which applies a linear transformation
    A common name is fully-connected layer, InnerProductLayer in caffe.

    The module should work with 2D input of shape (n_samples, n_feature).
    """
    def __init__(self, n_in, n_out):
        super(Linear, self).__init__()

        # This is a nice initialization
        stdv = 1./np.sqrt(n_in)
        self.W = np.random.uniform(-stdv, stdv, size = (n_out, n_in))
        self.b = np.random.uniform(-stdv, stdv, size = n_out)

        self.gradW = np.zeros_like(self.W)
        self.gradb = np.zeros_like(self.b)

    def updateOutput(self, input):
        # Your code goes here. ################################################
        self.output = input @ self.W.T + self.b
        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        self.gradInput = gradOutput @ self.W
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        # Your code goes here. ################################################
        self.gradW += gradOutput.T @ input
        self.gradb += np.sum(gradOutput, axis=0)
        

    def zeroGradParameters(self):
        self.gradW.fill(0)
        self.gradb.fill(0)

    def getParameters(self):
        return [self.W, self.b]

    def getGradParameters(self):
        return [self.gradW, self.gradb]

    def __repr__(self):
        s = self.W.shape
        q = 'Linear %d -> %d' %(s[1],s[0])
        return q

## 2. (0.2) SoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{softmax}(x)_i = \frac{\exp x_i} {\sum_j \exp x_j}$

Recall that $\text{softmax}(x) == \text{softmax}(x - \text{const})$. It makes possible to avoid computing exp() from large argument.

In [15]:
class SoftMax(Module):
    def __init__(self):
         super(SoftMax, self).__init__()

    def updateOutput(self, input):
        # start with normalization for numerical stability
        self.output = np.subtract(input, input.max(axis=1, keepdims=True))

        # Your code goes here. ################################################
        exp_x = np.exp(self.output)
        self.output = exp_x / np.sum(exp_x, axis=1, keepdims=True)

        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        s = np.sum(gradOutput * self.output, axis=1, keepdims=True)
        self.gradInput = self.output * (gradOutput - s)
        return self.gradInput

    def __repr__(self):
        return "SoftMax"

## 3. (0.2) LogSoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{logsoftmax}(x)_i = \log\text{softmax}(x)_i = x_i - \log {\sum_j \exp x_j}$

The main goal of this layer is to be used in computation of log-likelihood loss.

In [16]:
class LogSoftMax(Module):
    def __init__(self):
         super(LogSoftMax, self).__init__()

    def updateOutput(self, input):
        # start with normalization for numerical stability
        self.output = np.subtract(input, input.max(axis=1, keepdims=True))

        # Your code goes here. ################################################
        log_sum_exp = np.log(np.sum(np.exp(self.output), axis=1, keepdims=True))
        self.output = self.output - log_sum_exp
        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        softmax = np.exp(self.output)
        sum_grad = np.sum(gradOutput, axis=1, keepdims=True)
        self.gradInput = gradOutput - softmax * sum_grad
        return self.gradInput

    def __repr__(self):
        return "LogSoftMax"

## 4. (0.3) Batch normalization
One of the most significant recent ideas that impacted NNs a lot is [**Batch normalization**](http://arxiv.org/abs/1502.03167). The idea is simple, yet effective: the features should be whitened ($mean = 0$, $std = 1$) all the way through NN. This improves the convergence for deep models letting it train them for days but not weeks. **You are** to implement the first part of the layer: features normalization. The second part (`ChannelwiseScaling` layer) is implemented below.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

The layer should work as follows. While training (`self.training == True`) it transforms input as $$y = \frac{x - \mu}  {\sqrt{\sigma + \epsilon}}$$
where $\mu$ and $\sigma$ - mean and variance of feature values in **batch** and $\epsilon$ is just a small number for numericall stability. Also during training, layer should maintain exponential moving average values for mean and variance:
```
    self.moving_mean = self.moving_mean * alpha + batch_mean * (1 - alpha)
    self.moving_variance = self.moving_variance * alpha + batch_variance * (1 - alpha)
```
During testing (`self.training == False`) the layer normalizes input using moving_mean and moving_variance.

Note that decomposition of batch normalization on normalization itself and channelwise scaling here is just a common **implementation** choice. In general "batch normalization" always assumes normalization + scaling.

In [17]:
class BatchNormalization(Module):
    EPS = 1e-3
    def __init__(self, alpha = 0.):
        super(BatchNormalization, self).__init__()
        self.alpha = alpha
        self.moving_mean = None
        self.moving_variance = None

    def updateOutput(self, input):
        # Your code goes here. ################################################
        if self.training:
            batch_mean = np.mean(input, axis=0)
            batch_variance = np.var(input, axis=0)

            if self.moving_mean is None:
                self.moving_mean = batch_mean.copy()
                self.moving_variance = batch_variance.copy()
            else:
                self.moving_mean = self.moving_mean * self.alpha + batch_mean * (1 - self.alpha)
                self.moving_variance = self.moving_variance * self.alpha + batch_variance * (1 - self.alpha)

            self.output = (input - batch_mean) / np.sqrt(batch_variance + self.EPS)
        else:
            self.output = (input - self.moving_mean) / np.sqrt(self.moving_variance + self.EPS)

        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        if self.training:
            N = input.shape[0]
            mean = np.mean(input, axis=0)
            var = np.var(input, axis=0)

            std_inv = 1.0 / np.sqrt(var + self.EPS)
            x_hat = (input - mean) * std_inv

            self.gradInput = (1.0 / N) * std_inv * (N * gradOutput - np.sum(gradOutput, axis=0) - x_hat * np.sum(gradOutput * x_hat, axis=0))
        else:
            self.gradInput = gradOutput / np.sqrt(self.moving_variance + self.EPS)
        return self.gradInput

    def __repr__(self):
        return "BatchNormalization"

In [18]:
class ChannelwiseScaling(Module):
    """
       Implements linear transform of input y = \gamma * x + \beta
       where \gamma, \beta - learnable vectors of length x.shape[-1]
    """
    def __init__(self, n_out):
        super(ChannelwiseScaling, self).__init__()

        stdv = 1./np.sqrt(n_out)
        self.gamma = np.random.uniform(-stdv, stdv, size=n_out)
        self.beta = np.random.uniform(-stdv, stdv, size=n_out)

        self.gradGamma = np.zeros_like(self.gamma)
        self.gradBeta = np.zeros_like(self.beta)

    def updateOutput(self, input):
        self.output = input * self.gamma + self.beta
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * self.gamma
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        self.gradBeta = np.sum(gradOutput, axis=0)
        self.gradGamma = np.sum(gradOutput*input, axis=0)

    def zeroGradParameters(self):
        self.gradGamma.fill(0)
        self.gradBeta.fill(0)

    def getParameters(self):
        return [self.gamma, self.beta]

    def getGradParameters(self):
        return [self.gradGamma, self.gradBeta]

    def __repr__(self):
        return "ChannelwiseScaling"

<>:2: SyntaxWarning: invalid escape sequence '\g'
<>:2: SyntaxWarning: invalid escape sequence '\g'
/var/folders/s1/yzlnrsp13tvf9qnbwg952pk80000gn/T/ipykernel_4160/3092293347.py:2: SyntaxWarning: invalid escape sequence '\g'
  """


Practical notes. If BatchNormalization is placed after a linear transformation layer (including dense layer, convolutions, channelwise scaling) that implements function like `y = weight * x + bias`, than bias adding become useless and could be omitted since its effect will be discarded while batch mean subtraction. If BatchNormalization (followed by `ChannelwiseScaling`) is placed before a layer that propagates scale (including ReLU, LeakyReLU) followed by any linear transformation layer than parameter `gamma` in `ChannelwiseScaling` could be freezed since it could be absorbed into the linear transformation layer.

## 5. (0.3) Dropout
Implement [**dropout**](https://www.cs.toronto.edu/~hinton/absps/JMLRdropout.pdf). The idea and implementation is really simple: just multimply the input by $Bernoulli(p)$ mask. Here $p$ is probability of an element to be zeroed.

This has proven to be an effective technique for regularization and preventing the co-adaptation of neurons.

While training (`self.training == True`) it should sample a mask on each iteration (for every batch), zero out elements and multiply elements by $1 / (1 - p)$. The latter is needed for keeping mean values of features close to mean values which will be in test mode. When testing this module should implement identity transform i.e. `self.output = input`.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

In [19]:
class Dropout(Module):
    def __init__(self, p=0.5):
        super(Dropout, self).__init__()

        self.p = p
        self.mask = None

    def updateOutput(self, input):
        # Your code goes here. ################################################
        if self.training:
            self.mask = np.random.binomial(1, 1 - self.p, size = input.shape)
            self.output = input * self.mask / (1 - self.p)
        else:
            self.output = input
        return  self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        if self.training:
            self.gradInput = gradOutput * self.mask / (1 - self.p)
        else:
            self.gradInput = gradOutput
        return self.gradInput

    def __repr__(self):
        return "Dropout"

#6. (2.0) Conv2d
Implement [**Conv2d**](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html). Use only this list of parameters: (in_channels, out_channels, kernel_size, stride, padding, bias, padding_mode) and fix dilation=1 and groups=1.

In [20]:
class Conv2d(Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, padding=0, bias=True, padding_mode='zeros'):
        super(Conv2d, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.bias = bias
        self.padding_mode = padding_mode

    def updateOutput(self, input):
        # Your code goes here. ################################################
        if isinstance(self.kernel_size, int):
            kH, kW = self.kernel_size, self.kernel_size
        else:
            kH, kW = self.kernel_size

        if isinstance(self.stride, int):
            sH, sW = self.stride, self.stride
        else:
            sH, sW = self.stride

        H, W = input.shape[2], input.shape[3]

        if self.padding == 'same':
            outH = int(np.ceil(H / sH))
            outW = int(np.ceil(W / sW))

            pad_h_total = max((outH - 1) * sH + kH - H, 0)
            pad_w_total = max((outW - 1) * sW + kW - W, 0)

            pad_top = pad_h_total // 2
            pad_bottom = pad_h_total - pad_top
            pad_left = pad_w_total // 2
            pad_right = pad_w_total - pad_left
        else:
            if isinstance(self.padding, int):
                pH, pW = self.padding, self.padding
            else:
                pH, pW = self.padding

            pad_top = pad_bottom = pH
            pad_left = pad_right = pW

        if self.padding_mode == 'zeros':
            padded_input = np.pad(
                input,
                ((0, 0), (0, 0), (pad_top, pad_bottom), (pad_left, pad_right)),
                mode='constant',
                constant_values=0
            )
        elif self.padding_mode == 'replicate':
            padded_input = np.pad(
                input,
                ((0, 0), (0, 0), (pad_top, pad_bottom), (pad_left, pad_right)),
                mode='edge'
            )
        elif self.padding_mode == 'reflect':
            padded_input = np.pad(
                input,
                ((0, 0), (0, 0), (pad_top, pad_bottom), (pad_left, pad_right)),
                mode='reflect'
            )

        N, _, H_pad, W_pad = padded_input.shape
        outH = (H_pad - kH) // sH + 1
        outW = (W_pad - kW) // sW + 1

        self.output = np.zeros((N, self.out_channels, outH, outW), dtype=input.dtype)

        use_bias = isinstance(self.bias, np.ndarray)

        for n in range(N):
            for oc in range(self.out_channels):
                for i in range(outH):
                    h_start = i * sH
                    h_end = h_start + kH

                    for j in range(outW):
                        w_start = j * sW
                        w_end = w_start + kW

                        window = padded_input[n, :, h_start:h_end, w_start:w_end]
                        self.output[n, oc, i, j] = np.sum(window * self.weight[oc])

                        if use_bias:
                            self.output[n, oc, i, j] += self.bias[oc]

        return  self.output

    def updateGradInput(self, input, gradOutput):
    # Your code goes here. ################################################
        if isinstance(self.kernel_size, int):
            kH, kW = self.kernel_size, self.kernel_size
        else:
            kH, kW = self.kernel_size

        if isinstance(self.stride, int):
            sH, sW = self.stride, self.stride
        else:
            sH, sW = self.stride

        H, W = input.shape[2], input.shape[3]

        if self.padding == 'same':
            outH_same = int(np.ceil(H / sH))
            outW_same = int(np.ceil(W / sW))

            pad_h_total = max((outH_same - 1) * sH + kH - H, 0)
            pad_w_total = max((outW_same - 1) * sW + kW - W, 0)

            pad_top = pad_h_total // 2
            pad_bottom = pad_h_total - pad_top
            pad_left = pad_w_total // 2
            pad_right = pad_w_total - pad_left
        else:
            if isinstance(self.padding, int):
                pH, pW = self.padding, self.padding
            else:
                pH, pW = self.padding

            pad_top = pad_bottom = pH
            pad_left = pad_right = pW

        N, C_in, H, W = input.shape
        H_pad = H + pad_top + pad_bottom
        W_pad = W + pad_left + pad_right

        gradInput_padded = np.zeros((N, C_in, H_pad, W_pad), dtype=input.dtype)

        _, _, outH, outW = gradOutput.shape

        for n in range(N):
            for oc in range(self.out_channels):
                for i in range(outH):
                    h_start = i * sH
                    h_end = h_start + kH

                    for j in range(outW):
                        w_start = j * sW
                        w_end = w_start + kW

                        gradInput_padded[n, :, h_start:h_end, w_start:w_end] += (
                            gradOutput[n, oc, i, j] * self.weight[oc]
                        )

        if self.padding_mode == 'zeros':
            if pad_top == 0 and pad_bottom == 0 and pad_left == 0 and pad_right == 0:
                self.gradInput = gradInput_padded
            else:
                h_end = gradInput_padded.shape[2] - pad_bottom if pad_bottom > 0 else gradInput_padded.shape[2]
                w_end = gradInput_padded.shape[3] - pad_right if pad_right > 0 else gradInput_padded.shape[3]
                self.gradInput = gradInput_padded[:, :, pad_top:h_end, pad_left:w_end]
        else:
            self.gradInput = np.zeros_like(input)

            for pi in range(H_pad):
                if self.padding_mode == 'replicate':
                    ii = min(max(pi - pad_top, 0), H - 1)
                else:  # reflect
                    ii = pi - pad_top
                    while ii < 0 or ii >= H:
                        if ii < 0:
                            ii = -ii
                        if ii >= H:
                            ii = 2 * H - ii - 2

                for pj in range(W_pad):
                    if self.padding_mode == 'replicate':
                        jj = min(max(pj - pad_left, 0), W - 1)
                    else:  # reflect
                        jj = pj - pad_left
                        while jj < 0 or jj >= W:
                            if jj < 0:
                                jj = -jj
                            if jj >= W:
                                jj = 2 * W - jj - 2

                    self.gradInput[:, :, ii, jj] += gradInput_padded[:, :, pi, pj]

        return self.gradInput

    def __repr__(self):
        return "Conv2d"

#7. (0.5) Implement [**MaxPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) and [**AvgPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html). Use only parameters like kernel_size, stride, padding (negative infinity for maxpool and zero for avgpool) and other parameters fixed as in framework.

In [21]:
class MaxPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(MaxPool2d, self).__init__()

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def updateOutput(self, input):
        # Your code goes here. ################################################
        if isinstance(self.kernel_size, int):
            kH, kW = self.kernel_size, self.kernel_size
        else:
            kH, kW = self.kernel_size

        if isinstance(self.stride, int):
            sH, sW = self.stride, self.stride
        else:
            sH, sW = self.stride

        if isinstance(self.padding, int):
            pH, pW = self.padding, self.padding
        else:
            pH, pW = self.padding

        padded_input = np.pad(
            input,
            ((0, 0), (0, 0), (pH, pH), (pW, pW)),
            mode='constant',
            constant_values=-np.inf
        )

        N, C, H_pad, W_pad = padded_input.shape
        outH = (H_pad - kH) // sH + 1
        outW = (W_pad - kW) // sW + 1

        self.output = np.zeros((N, C, outH, outW), dtype=input.dtype)

        for n in range(N):
            for c in range(C):
                for i in range(outH):
                    h_start = i * sH
                    h_end = h_start + kH

                    for j in range(outW):
                        w_start = j * sW
                        w_end = w_start + kW

                        window = padded_input[n, c, h_start:h_end, w_start:w_end]
                        self.output[n, c, i, j] = np.max(window)

        return  self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        if isinstance(self.kernel_size, int):
            kH, kW = self.kernel_size, self.kernel_size
        else:
            kH, kW = self.kernel_size

        if isinstance(self.stride, int):
            sH, sW = self.stride, self.stride
        else:
            sH, sW = self.stride

        if isinstance(self.padding, int):
            pH, pW = self.padding, self.padding
        else:
            pH, pW = self.padding

        padded_input = np.pad(
            input,
            ((0, 0), (0, 0), (pH, pH), (pW, pW)),
            mode='constant',
            constant_values=-np.inf
        )

        gradInput_padded = np.zeros_like(padded_input)

        N, C, outH, outW = gradOutput.shape

        for n in range(N):
            for c in range(C):
                for i in range(outH):
                    h_start = i * sH
                    h_end = h_start + kH

                    for j in range(outW):
                        w_start = j * sW
                        w_end = w_start + kW

                        window = padded_input[n, c, h_start:h_end, w_start:w_end]
                        max_value = np.max(window)
                        mask = (window == max_value)

                        gradInput_padded[n, c, h_start:h_end, w_start:w_end] += (
                            gradOutput[n, c, i, j] * mask
                        )

        if pH == 0 and pW == 0:
            self.gradInput = gradInput_padded
        else:
            h_end = gradInput_padded.shape[2] - pH if pH > 0 else gradInput_padded.shape[2]
            w_end = gradInput_padded.shape[3] - pW if pW > 0 else gradInput_padded.shape[3]
            self.gradInput = gradInput_padded[:, :, pH:h_end, pW:w_end]

        return self.gradInput

    def __repr__(self):
        return "MaxPool2d"

class AvgPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(AvgPool2d, self).__init__()

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def updateOutput(self, input):
        # Your code goes here. ################################################
        if isinstance(self.kernel_size, int):
            kH, kW = self.kernel_size, self.kernel_size
        else:
            kH, kW = self.kernel_size

        if isinstance(self.stride, int):
            sH, sW = self.stride, self.stride
        else:
            sH, sW = self.stride

        if isinstance(self.padding, int):
            pH, pW = self.padding, self.padding
        else:
            pH, pW = self.padding

        padded_input = np.pad(
            input,
            ((0, 0), (0, 0), (pH, pH), (pW, pW)),
            mode='constant',
            constant_values=0
        )

        N, C, H_pad, W_pad = padded_input.shape
        outH = (H_pad - kH) // sH + 1
        outW = (W_pad - kW) // sW + 1

        self.output = np.zeros((N, C, outH, outW), dtype=input.dtype)

        for n in range(N):
            for c in range(C):
                for i in range(outH):
                    h_start = i * sH
                    h_end = h_start + kH

                    for j in range(outW):
                        w_start = j * sW
                        w_end = w_start + kW

                        window = padded_input[n, c, h_start:h_end, w_start:w_end]
                        self.output[n, c, i, j] = np.mean(window)

        return  self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        if isinstance(self.kernel_size, int):
            kH, kW = self.kernel_size, self.kernel_size
        else:
            kH, kW = self.kernel_size

        if isinstance(self.stride, int):
            sH, sW = self.stride, self.stride
        else:
            sH, sW = self.stride

        if isinstance(self.padding, int):
            pH, pW = self.padding, self.padding
        else:
            pH, pW = self.padding

        padded_input = np.pad(
            input,
            ((0, 0), (0, 0), (pH, pH), (pW, pW)),
            mode='constant',
            constant_values=0
        )

        gradInput_padded = np.zeros_like(padded_input)

        N, C, outH, outW = gradOutput.shape
        coeff = 1.0 / (kH * kW)

        for n in range(N):
            for c in range(C):
                for i in range(outH):
                    h_start = i * sH
                    h_end = h_start + kH

                    for j in range(outW):
                        w_start = j * sW
                        w_end = w_start + kW

                        gradInput_padded[n, c, h_start:h_end, w_start:w_end] += (
                            gradOutput[n, c, i, j] * coeff
                        )

        if pH == 0 and pW == 0:
            self.gradInput = gradInput_padded
        else:
            h_end = gradInput_padded.shape[2] - pH if pH > 0 else gradInput_padded.shape[2]
            w_end = gradInput_padded.shape[3] - pW if pW > 0 else gradInput_padded.shape[3]
            self.gradInput = gradInput_padded[:, :, pH:h_end, pW:w_end]

        return self.gradInput

    def __repr__(self):
        return "AvgPool2d"

#8. (0.3) Implement **GlobalMaxPool2d** and **GlobalAvgPool2d**. They do not have testing and parameters are up to you but they must aggregate information within channels. Write test functions for these layers on your own.

In [22]:
class GlobalMaxPool2d(Module):
    def __init__(self):
        super(GlobalMaxPool2d, self).__init__()

    def updateOutput(self, input):
        self.output = np.max(input, axis=(2, 3))
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = np.zeros_like(input)

        max_vals = np.max(input, axis=(2, 3), keepdims=True)
        mask = (input == max_vals)

        self.gradInput = mask * gradOutput[:, :, None, None]
        return self.gradInput

    def __repr__(self):
        return "GlobalMaxPool2d"


class GlobalAvgPool2d(Module):
    def __init__(self):
        super(GlobalAvgPool2d, self).__init__()

    def updateOutput(self, input):
        self.output = np.mean(input, axis=(2, 3))
        return self.output

    def updateGradInput(self, input, gradOutput):
        N, C, H, W = input.shape
        self.gradInput = np.ones_like(input) * (gradOutput[:, :, None, None] / (H * W))
        return self.gradInput

    def __repr__(self):
        return "GlobalAvgPool2d"

#9. (0.2) Implement [**Flatten**](https://pytorch.org/docs/stable/generated/torch.flatten.html)

In [23]:
class Flatten(Module):
    def __init__(self, start_dim=0, end_dim=-1):
        super(Flatten, self).__init__()

        self.start_dim = start_dim
        self.end_dim = end_dim

    def updateOutput(self, input):
        # Your code goes here. ################################################
        ndim = input.ndim

        start_dim = self.start_dim if self.start_dim >= 0 else self.start_dim + ndim
        end_dim = self.end_dim if self.end_dim >= 0 else self.end_dim + ndim

        shape = input.shape
        flattened_dim = np.prod(shape[start_dim:end_dim + 1])

        new_shape = shape[:start_dim] + (flattened_dim,) + shape[end_dim + 1:]
        self.output = input.reshape(new_shape)

        return  self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        self.gradInput = gradOutput.reshape(input.shape)
        return self.gradInput

    def __repr__(self):
        return "Flatten"

# Activation functions

Here's the complete example for the **Rectified Linear Unit** non-linearity (aka **ReLU**):

In [24]:
class ReLU(Module):
    def __init__(self):
         super(ReLU, self).__init__()

    def updateOutput(self, input):
        self.output = np.maximum(input, 0)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = np.multiply(gradOutput , input > 0)
        return self.gradInput

    def __repr__(self):
        return "ReLU"

## 10. (0.1) Leaky ReLU
Implement [**Leaky Rectified Linear Unit**](http://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29%23Leaky_ReLUs). Expriment with slope.

In [25]:
class LeakyReLU(Module):
    def __init__(self, slope = 0.03):
        super(LeakyReLU, self).__init__()

        self.slope = slope

    def updateOutput(self, input):
        # Your code goes here. ################################################
        self.output = np.where(input > 0, input, self.slope * input)
        return  self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        self.gradInput = gradOutput * np.where(input > 0, 1, self.slope)
        return self.gradInput

    def __repr__(self):
        return "LeakyReLU"

## 11. (0.1) ELU
Implement [**Exponential Linear Units**](http://arxiv.org/abs/1511.07289) activations.

In [26]:
class ELU(Module):
    def __init__(self, alpha = 1.0):
        super(ELU, self).__init__()

        self.alpha = alpha

    def updateOutput(self, input):
        # Your code goes here. ################################################
        self.output = np.where(input > 0, input, self.alpha * (np.exp(input) - 1))
        return  self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        self.gradInput = gradOutput * np.where(input > 0, 1, self.alpha * np.exp(input))
        return self.gradInput

    def __repr__(self):
        return "ELU"

## 12. (0.1) SoftPlus
Implement [**SoftPlus**](https://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29) activations. Look, how they look a lot like ReLU.

In [27]:
class SoftPlus(Module):
    def __init__(self):
        super(SoftPlus, self).__init__()

    def updateOutput(self, input):
        # Your code goes here. ################################################
        self.output = np.log1p(np.exp(-np.abs(input))) + np.maximum(input, 0)
        return  self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        sigmoid = 1 / (1 + np.exp(-input))
        self.gradInput = gradOutput * sigmoid
        return self.gradInput

    def __repr__(self):
        return "SoftPlus"

#13. (0.2) Gelu
Implement [**Gelu**](https://pytorch.org/docs/stable/generated/torch.nn.GELU.html) activations.


In [28]:
import math

class Gelu(Module):
    def __init__(self):
        super(Gelu, self).__init__()

    def updateOutput(self, input):
        # Your code goes here. ################################################
        erf_input = np.vectorize(math.erf)(input / math.sqrt(2.0))
        self.output = 0.5 * input * (1.0 + erf_input)
        return  self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        erf_input = np.vectorize(math.erf)(input / math.sqrt(2.0))
        gelu_grad = 0.5 * (1.0 + erf_input) + input * np.exp(-(input ** 2) / 2.0) / math.sqrt(2.0 * math.pi)
        self.gradInput = gradOutput * gelu_grad
        return self.gradInput

    def __repr__(self):
        return "Gelu"

# Criterions

Criterions are used to score the models answers.

In [29]:
class Criterion(object):
    def __init__ (self):
        self.output = None
        self.gradInput = None

    def forward(self, input, target):
        """
            Given an input and a target, compute the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateOutput`.
        """
        return self.updateOutput(input, target)

    def backward(self, input, target):
        """
            Given an input and a target, compute the gradients of the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateGradInput`.
        """
        return self.updateGradInput(input, target)

    def updateOutput(self, input, target):
        """
        Function to override.
        """
        return self.output

    def updateGradInput(self, input, target):
        """
        Function to override.
        """
        return self.gradInput

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Criterion"

The **MSECriterion**, which is basic L2 norm usually used for regression, is implemented here for you.
- input:   **`batch_size x n_feats`**
- target: **`batch_size x n_feats`**
- output: **scalar**

In [30]:
class MSECriterion(Criterion):
    def __init__(self):
        super(MSECriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = np.sum(np.power(input - target,2)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput  = (input - target) * 2 / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "MSECriterion"

## 14. (0.2) Negative LogLikelihood criterion (numerically unstable)
You task is to implement the **ClassNLLCriterion**. It should implement [multiclass log loss](http://scikit-learn.org/stable/modules/model_evaluation.html#log-loss). Nevertheless there is a sum over `y` (target) in that formula,
remember that targets are one-hot encoded. This fact simplifies the computations a lot. Note, that criterions are the only places, where you divide by batch size. Also there is a small hack with adding small number to probabilities to avoid computing log(0).
- input:   **`batch_size x n_feats`** - probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**



In [31]:
class ClassNLLCriterionUnstable(Criterion):
    EPS = 1e-15
    def __init__(self):
        a = super(ClassNLLCriterionUnstable, self)
        super(ClassNLLCriterionUnstable, self).__init__()

    def updateOutput(self, input, target):

        # Use this trick to avoid numerical errors
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)

        # Your code goes here. ################################################
        self.output = -np.sum(target * np.log(input_clamp)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):

        # Use this trick to avoid numerical errors
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)

        # Your code goes here. ################################################
        self.gradInput = -(target / input_clamp) / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterionUnstable"

## 15. (0.3) Negative LogLikelihood criterion (numerically stable)
- input:   **`batch_size x n_feats`** - log probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**

Task is similar to the previous one, but now the criterion input is the output of log-softmax layer. This decomposition allows us to avoid problems with computation of forward and backward of log().

In [32]:
class ClassNLLCriterion(Criterion):
    def __init__(self):
        a = super(ClassNLLCriterion, self)
        super(ClassNLLCriterion, self).__init__()

    def updateOutput(self, input, target):
        # Your code goes here. ################################################
        self.output = -np.sum(target * input) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        # Your code goes here. ################################################
        self.gradInput = -target / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterion"

1-я часть задания: реализация слоев, лосей и функций активации - 5 баллов. \\
2-я часть задания: реализация моделей на своих классах. Что должно быть:
  1. Выберите оптимизатор и реализуйте его, чтоб он работал с вами классами. - 1 балл.
  2. Модель для задачи мультирегрессии на выбраных вами данных. Использовать FCNN, dropout, batchnorm, MSE. Пробуйте различные фукнции активации. Для первой модели попробуйте большую, среднюю и маленькую модель. - 1 балл.
  3. Модель для задачи мультиклассификации на MNIST. Использовать свёртки, макспулы, флэттэны, софтмаксы - 1 балла.
  4. Автоэнкодер для выбранных вами данных. Должен быть на свёртках и полносвязных слоях, дропаутах, батчнормах и тд. - 2 балла. \\

Дополнительно в оценке каждой модели будет учитываться:
1. Наличие правильно выбранной метрики и лосс функции.
2. Отрисовка графиков лосей и метрик на трейне-валидации. Проверка качества модели на тесте.
3. Наличие шедулера для lr.
4. Наличие вормапа.
5. Наличие механизма ранней остановки и сохранение лучшей модели.
6. Свитч лося (метрики) и оптимайзера.

задание 1


In [33]:
import numpy as np

class Adam(object):
    def __init__(self, model, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.model = model
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0

        self.params = self._unpack(self.model.getParameters())
        self.gradParams = self._unpack(self.model.getGradParameters())

        self.m = [np.zeros_like(p) for p in self.params]
        self.v = [np.zeros_like(p) for p in self.params]

    def _unpack(self, params):
        result = []
        for x in params:
            if isinstance(x, list):
                result += self._unpack(x)
            else:
                result.append(x)
        return result

    def zeroGradParameters(self):
        self.model.zeroGradParameters()

    def step(self):
        self.t += 1

        self.params = self._unpack(self.model.getParameters())
        self.gradParams = self._unpack(self.model.getGradParameters())

        for i in range(len(self.params)):
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * self.gradParams[i]
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (self.gradParams[i] ** 2)

            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)

            self.params[i] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

задание 2

In [34]:
import matplotlib.pyplot as plt

from sklearn.datasets import load_linnerud
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [35]:
data = load_linnerud()

X = data.data
y = data.target

In [36]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42
)

In [37]:
# Нормализация признаков
x_scaler = StandardScaler()
X_train = x_scaler.fit_transform(X_train)
X_val = x_scaler.transform(X_val)
X_test = x_scaler.transform(X_test)

# Нормализация таргетов
y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train)
y_val = y_scaler.transform(y_val)
y_test = y_scaler.transform(y_test)

In [38]:
def build_regression_model(input_dim, output_dim, size='small', activation='relu', dropout_p=0.1):
    model = Sequential()

    if size == 'small':
        hidden_dims = [32, 16]
    elif size == 'medium':
        hidden_dims = [64, 32, 16]
    elif size == 'large':
        hidden_dims = [128, 64, 32, 16]

    prev_dim = input_dim

    for h in hidden_dims:
        model.add(Linear(prev_dim, h))
        model.add(BatchNormalization())
        model.add(get_activation(activation))
        model.add(Dropout(dropout_p))
        prev_dim = h

    model.add(Linear(prev_dim, output_dim))

    return model

In [39]:
def mse_score(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def mae_score(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def r2_score_np(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2, axis=0)
    ss_tot = np.sum((y_true - np.mean(y_true, axis=0)) ** 2, axis=0)
    return np.mean(1 - ss_res / (ss_tot + 1e-12))

In [40]:
def small(input_dim, output_dim):
    model = Sequential()

    model.add(Linear(input_dim, 32))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(Dropout(0.1))

    model.add(Linear(32, 16))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(Dropout(0.1))

    model.add(Linear(16, output_dim))

    return model

In [41]:
def medium(input_dim, output_dim):
    model = Sequential()

    model.add(Linear(input_dim, 64))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(Dropout(0.1))

    model.add(Linear(64, 32))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(Dropout(0.1))

    model.add(Linear(32, 16))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(Dropout(0.1))

    model.add(Linear(16, output_dim))

    return model

In [42]:
def large(input_dim, output_dim):
    model = Sequential()

    model.add(Linear(input_dim, 128))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(Dropout(0.1))

    model.add(Linear(128, 64))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(Dropout(0.1))

    model.add(Linear(64, 32))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(Dropout(0.1))

    model.add(Linear(32, 16))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(Dropout(0.1))

    model.add(Linear(16, output_dim))

    return model

дальше не успела(